# Digital Divide Cusco — Geospatial Raster Analysis
**Course:** Python Programming — Applied Data Science  
**Task:** Territorial Digital Divide: Geospatial Raster Analysis  
**Region:** Cusco, Peru

## Step 0 — Environment Setup

In [ ]:
# Uncomment to install in Google Colab
# !pip install rasterio==1.4.3 numpy==2.0.2 matplotlib==3.10.0 scipy==1.15.3 seaborn==0.13.2 pandas==2.2.2

In [1]:
import rasterio
import rasterio.warp
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.transform import array_bounds
from rasterio.plot import plotting_extent

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
import scipy
from scipy.ndimage import gaussian_filter
from scipy import stats
import os

# Plot style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Paths
DATA_DIR   = '../data'
OUTPUT_DIR = '../output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

VNL_PATH  = os.path.join(DATA_DIR, 'VNL_cusco_2025.tif')
CONN_PATH = os.path.join(DATA_DIR, 'kernel_cobmovil2019_50m.tif')

print('Libraries loaded successfully.')
print()
print(f'  rasterio  : {rasterio.__version__}')
print(f'  numpy     : {np.__version__}')
print(f'  pandas    : {pd.__version__}')
print(f'  matplotlib: {plt.matplotlib.__version__}')
print(f'  scipy     : {scipy.__version__}')
print(f'  seaborn   : {sns.__version__}')
print()
print(f'  VNL path  : {VNL_PATH}')
print(f'  CONN path : {CONN_PATH}')
print(f'  Output dir: {OUTPUT_DIR}')


Libraries loaded successfully.

  rasterio  : 1.4.3
  numpy     : 2.0.2
  pandas    : 2.2.2
  matplotlib: 3.10.0
  scipy     : 1.15.3
  seaborn   : 0.13.2

  VNL path  : ../data\VNL_cusco_2025.tif
  CONN path : ../data\kernel_cobmovil2019_50m.tif
  Output dir: ../output


## Step 1 — Raster Loading and Inspection

In [2]:
def inspect_raster(path, name):
    with rasterio.open(path) as src:
        data    = src.read(1).astype(np.float64)
        nodata  = src.nodata
        res_deg = abs(src.transform.a)
        res_km  = res_deg * 111

        if nodata is not None:
            valid_mask = data != nodata
        else:
            valid_mask = np.ones(data.shape, dtype=bool)

        valid_pixels = data[valid_mask]
        n_valid      = valid_pixels.size
        n_total      = data.size

        print(f'=== {name} ===')
        print(f'  CRS              : {src.crs}')
        print(f'  Shape            : {src.height} rows x {src.width} cols')
        print(f'  Bands            : {src.count}')
        print(f'  Data type        : {src.dtypes[0]}')
        print(f'  NoData value     : {nodata}')
        print(f'  Bounding box     : {src.bounds}')
        print(f'  Pixel resolution : {res_deg:.6f} deg (~{res_km:.2f} km)')
        print(f'  Valid pixels     : {n_valid:,} / {n_total:,} ({100*n_valid/n_total:.1f}%)')
        print(f'  Value range      : min={valid_pixels.min():.6f}  max={valid_pixels.max():.6f}')
        print()

        return src.meta.copy(), src.transform, src.crs, src.bounds

print('--- Loading rasters ---')
print()
vnl_meta,  vnl_transform,  vnl_crs,  vnl_bounds  = inspect_raster(VNL_PATH,  'VNL_cusco_2025.tif  (Nighttime Lights)')
conn_meta, conn_transform, conn_crs, conn_bounds = inspect_raster(CONN_PATH, 'kernel_cobmovil2019_50m.tif  (Mobile Coverage)')

print('Key observation:')
print('  - VNL is in EPSG:4326 (geographic degrees) -- no reprojection needed.')
print('  - Connectivity is in EPSG:32719 (UTM meters) -- must reproject to EPSG:4326.')
print('  - VNL resolution ~0.004167 deg (~463 m); connectivity resolution = 50 m.')
print('  - After reprojection, connectivity will be resampled to the VNL grid.')


--- Loading rasters ---

=== VNL_cusco_2025.tif  (Nighttime Lights) ===
  CRS              : EPSG:4326
  Shape            : 1081 rows x 961 cols
  Bands            : 1
  Data type        : float32
  NoData value     : None
  Bounding box     : BoundingBox(left=-74.00208248534999, bottom=-15.50208405735, right=-69.99791578664998, top=-10.99791735465)
  Pixel resolution : 0.004167 deg (~0.46 km)
  Valid pixels     : 1,038,841 / 1,038,841 (100.0%)
  Value range      : min=-1.500000  max=1254.614502

=== kernel_cobmovil2019_50m.tif  (Mobile Coverage) ===
  CRS              : EPSG:32719
  Shape            : 6116 rows x 7754 cols
  Bands            : 1
  Data type        : float32
  NoData value     : -3.4028234663852886e+38
  Bounding box     : BoundingBox(left=-43080.11101302641, bottom=8337100.058809407, right=344619.88898697356, top=8642900.058809407)
  Pixel resolution : 50.000000 deg (~5550.00 km)
  Valid pixels     : 47,423,464 / 47,423,464 (100.0%)
  Value range      : min=0.000000  